# AI Person Detection (YOLOv8) — Colab GPU T4 (Complete: Inference → Fine-tune → Inference)

Notebook นี้ออกแบบให้ “สมบูรณ์” สำหรับงานสาธิต:
1) ตรวจจับบุคคลจาก **ภาพ** (Base model)
2) ตรวจจับบุคคลจาก **วิดีโอ** + แปลงผลลัพธ์เป็น **MP4** (ดูใน Colab ได้)
3) **Fine-tune** ด้วย public dataset (COCO8) แบบไม่ค้าง (unzip -o)
4) โหลด **โมเดลที่ฝึกแล้ว (best.pt)** และรันตรวจจับซ้ำ (After fine-tune)
5) ดาวน์โหลดไฟล์สำคัญ (best.pt และ mp4 output)

ก่อนเริ่ม: **Runtime → Change runtime type → GPU** (ควรเป็น Tesla T4)


In [ ]:
# 1) Install dependencies
!pip -q install ultralytics opencv-python
import ultralytics
ultralytics.checks()


In [ ]:
# 2) Check GPU (ต้องเห็น Tesla T4)
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# 3) Load base model (YOLOv8n)
from ultralytics import YOLO
base_model = YOLO('yolov8n.pt')
base_model


## 4) Inference (Image) — Base model
อัปโหลดภาพของคุณเอง (แนะนำ: ไม่ระบุตัวตน/เบลอหน้า)


In [ ]:
from google.colab import files
uploaded = files.upload()
img_path = next(iter(uploaded.keys()))
img_path


In [ ]:
# Predict person only (COCO class 0)
base_img_results = base_model.predict(source=img_path, classes=[0], conf=0.25, device=0, save=True)

import matplotlib.pyplot as plt
base_img_plot = base_img_results[0].plot()
plt.figure(figsize=(10,6))
plt.title('Base model (yolov8n.pt)')
plt.imshow(base_img_plot)
plt.axis('off')
plt.show()


## 5) Inference (Video) — Base model (Convert to MP4 + Download)
อัปโหลดวิดีโอสั้น 10–30 วินาที เพื่อให้รันเร็ว


In [ ]:
uploaded = files.upload()
vid_path = next(iter(uploaded.keys()))
vid_path


In [ ]:
# Run detection on video
base_vid_results = base_model.predict(source=vid_path, classes=[0], conf=0.25, device=0, save=True)

import glob, os
from IPython.display import Video, display

# Find latest output video saved by Ultralytics
cands = sorted(glob.glob('runs/detect/predict*/**/*.*', recursive=True))
base_video_raw = None
for p in reversed(cands):
    if os.path.splitext(p)[1].lower() in ['.mp4', '.avi', '.mov', '.mkv']:
        base_video_raw = p
        break
print('Base raw output:', base_video_raw)

# Convert to MP4 for web preview
base_video_mp4 = 'base_output.mp4'
!ffmpeg -y -i "{base_video_raw}" -vcodec libx264 -acodec aac {base_video_mp4}

display(Video(base_video_mp4, embed=True))


## 6) Fine-tune YOLOv8 with Public Dataset (COCO8)
ส่วนนี้ใช้ COCO8 (public dataset ขนาดเล็ก) เพื่อสาธิต training flow ให้จบเร็ว

หมายเหตุ: ใช้ `unzip -o` เพื่อไม่ให้ค้างถาม replace


In [ ]:
from ultralytics.utils.downloads import download
download('https://github.com/ultralytics/assets/releases/download/v0.0.0/coco8.zip')
!unzip -o -q coco8.zip
!ls


In [ ]:
# Train (แนะนำ epochs=1–3 สำหรับเดโม)
ft_model = YOLO('yolov8n.pt')
ft_model.train(data='coco8.yaml', epochs=3, imgsz=640, batch=16, device=0)


In [ ]:
# Locate best.pt (เผื่อ path เปลี่ยนตามเวอร์ชัน)
import glob
best_candidates = sorted(glob.glob('runs/**/weights/best.pt', recursive=True))
print('Found best.pt candidates:')
for p in best_candidates[-5:]:
    print(' -', p)

best_pt = best_candidates[-1] if best_candidates else None
print('\nUsing best.pt:', best_pt)


## 7) Inference AFTER Fine-tune (ใช้ best.pt)
รันตรวจจับซ้ำกับภาพ/วิดีโอเดิม เพื่อให้เห็นผล “ก่อน–หลัง”


In [ ]:
assert best_pt is not None, 'ไม่พบ best.pt — กรุณาดูว่า train สำเร็จหรือไม่'
tuned_model = YOLO(best_pt)
tuned_model


In [ ]:
# Inference on the same image (after fine-tune)
tuned_img_results = tuned_model.predict(source=img_path, classes=[0], conf=0.25, device=0, save=True)

import matplotlib.pyplot as plt
tuned_img_plot = tuned_img_results[0].plot()
plt.figure(figsize=(10,6))
plt.title('Tuned model (best.pt)')
plt.imshow(tuned_img_plot)
plt.axis('off')
plt.show()


In [ ]:
# Side-by-side comparison (Image)
import matplotlib.pyplot as plt

plt.figure(figsize=(14,6))
plt.subplot(1,2,1)
plt.title('Base (yolov8n.pt)')
plt.imshow(base_img_plot)
plt.axis('off')

plt.subplot(1,2,2)
plt.title('After Fine-tune (best.pt)')
plt.imshow(tuned_img_plot)
plt.axis('off')
plt.show()


In [ ]:
# Inference on the same video (after fine-tune) + convert to MP4
tuned_vid_results = tuned_model.predict(source=vid_path, classes=[0], conf=0.25, device=0, save=True)

import glob, os
from IPython.display import Video, display

cands = sorted(glob.glob('runs/detect/predict*/**/*.*', recursive=True))
tuned_video_raw = None
for p in reversed(cands):
    if os.path.splitext(p)[1].lower() in ['.mp4', '.avi', '.mov', '.mkv']:
        tuned_video_raw = p
        break
print('Tuned raw output:', tuned_video_raw)

tuned_video_mp4 = 'tuned_output.mp4'
!ffmpeg -y -i "{tuned_video_raw}" -vcodec libx264 -acodec aac {tuned_video_mp4}
display(Video(tuned_video_mp4, embed=True))


## 8) หมายเหตุสำคัญสำหรับการสาธิต
- COCO8 เป็น dataset ตัวอย่างหลายคลาส; เรากรองเฉพาะ person ด้วย `classes=[0]` ตอน predict
- เป้าหมายของการ fine-tune ในเดโม คือให้ผู้เรียนเห็น workflow: dataset → train → best.pt → ใช้งานต่อ
